<div dir="rtl">
<h1>هیچ عددی نباید به Head اشتباه برود</h1>
<p>درس 41 از 76 · چگونه C ویژگی را میان چند سر تقسیم کنیم؟ · <code dir="ltr">35-split-heads</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-01/35-split-heads.html">📖 بازگشت به همین درس</a></p>
<p>تقسیم ویژگی‌ها را با نشانی عناصر، نه فقط Shape، تأیید کنید.</p><p>پیش‌نیاز: reshape و Transpose و رابطهٔ C=H*D را مرور کنید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>در x شکل (2,5,12) با H=3، عنصر x[1,3,9] بعد از تقسیم در کدام Head و کدام ویژگی آن است؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from mini_gpt.config import ModelConfig
from mini_gpt.attention import CausalSelfAttention
x = torch.arange(120.).reshape(2,5,12)
print('tracked value:',x[1,3,9].item())

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع split_heads(x, heads) را بنویسید: (B,T,C) به (B,H,T,D). اگر C بر H بخش‌پذیر نیست یا H مثبت نیست، ValueError بدهید. داده را حذف یا با صفر پر نکنید.</p>
</div>

In [ ]:
def split_heads(x, heads):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = split_heads(x,3)
    if result is None: return False
    assert result.shape == (2,3,5,4)
    assert result[1,2,3,1] == x[1,3,9]
    for H in (1,2,4):
        out = split_heads(x,H); D = 12//H
        for h in range(H):
            assert torch.equal(out[:,h],x[:,:,h*D:(h+1)*D])
    for invalid in (0,-1,5):
        try: split_heads(x,invalid)
        except ValueError: pass
        else: raise AssertionError('reject invalid C/H')
    attention = CausalSelfAttention(ModelConfig(12,8,12,3,1,0.)).eval()
    trace = {}; attention(x,trace=trace)
    q = attention.qkv(x).chunk(3,-1)[0]
    torch.testing.assert_close(split_heads(q,3),trace['q'])
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط H را میان ۱، ۳ و ۴ تغییر دهید و C=12 را ثابت نگه دارید. D و تعداد جدول‌های Attention چه می‌شوند؟</p>
</div>

In [ ]:
for H in (1,3,4):
    config = ModelConfig(12,8,12,H,1,0.)
    module = CausalSelfAttention(config)
    print('H,D,parameters:',H,12//H,sum(p.numel() for p in module.parameters()))

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>reshape مستقیم به (B,H,T,D) Shape درست ولی نشانی اشتباه می‌سازد. تابع repair_split(x, heads) را اصلاح کنید؛ ورودی‌های این بخش تقسیم‌پذیرند.</p>
</div>

In [ ]:
wrong = x.reshape(2,3,5,4)
print('wrong value:',wrong[1,2,3,1].item(),'expected:',x[1,3,9].item())

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def repair_split(x, heads):
    # TODO
    return None

In [ ]:
def test_repair():
    result = repair_split(x,3)
    if result is None: return False
    assert torch.equal(result[:,1],x[:,:,4:8])
    y = torch.arange(48.).reshape(1,4,12)
    assert torch.equal(repair_split(y,2)[:,1],y[:,:,6:])
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>تابع شما با q ثبت‌شده در trace واقعی CausalSelfAttention مقایسه شد. تقسیم پس از Projection انجام می‌شود؛ هر Head همهٔ موقعیت‌های مجاز را می‌بیند، نه یک قطعهٔ متن جدا.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>کدام assertion می‌توانست خطایی را بگیرد که آزمون Shape از آن عبور می‌کرد؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-01/35-split-heads.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/35-split-heads.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>